# LIME From Scratch (regression)
**Dr. Dave Wanik - OPIM 5512: Data Science Using Python - University of Connecticut**

---------------------------------

Last lecture we explained one prediction with **SHAP** — game theory, fair credit, every dollar accounted for. **LIME** answers the *same* local question ("why *this* prediction?") with a completely different and wonderfully simple trick:

> **Zoom in.** Any curvy black-box model, if you look closely enough at one point, looks *almost straight*. So fit a tiny **transparent** model — a plain linear regression — to the black box's behavior **right around that one point**. The slopes of that little local line *are* the explanation.

LIME = **L**ocal **I**nterpretable **M**odel-agnostic **E**xplanations (Ribeiro, Singh & Guestrin, 2016). "Model-agnostic" because we only ever call `model.predict()` — LIME never looks inside. As always: **we build it by hand first**, then in a few lines of Python, *then* check it against the `lime` package.

## The whole idea in one picture: zoom in until it's a line

A parabola is curvy. But stand on it at $x=3$ and look only at your feet — the curve looks like a **straight line** with a slope. That slope is how the output changes *right here*, and it's exactly what a calculus teacher calls the **derivative**.

LIME is that idea for *any* model and *any* number of features:

1. Pick the row $x_0$ you want to explain.
2. **Perturb** it — jiggle the feature values to create a cloud of nearby fake rows.
3. Ask the black box for its prediction on each fake row (`model.predict`).
4. **Weight** each fake row by how *close* it is to $x_0$ — near points matter, far points barely count.
5. Fit a **weighted linear regression** to (fake rows → black-box predictions).
6. Read off the **coefficients**. Those are your local feature importances — "around here, bumping this feature up moves the prediction by *this much*."

That's the entire algorithm. Let's earn it on the whiteboard.

## Step 1 (by hand) — a curvy black box and a local line

Pretend our black-box model is

$$ f(x) = x^2 $$

and we want to explain its prediction at $x_0 = 3$ (so $f(3) = 9$). We don't get to peek at the formula — we can only **query** it. So we query three nearby points and fit a line through the answers.

| perturbed $x$ | black-box $y = f(x)$ |
|:---:|:---:|
| 2 | 4 |
| 3 | 9 |
| 4 | 16 |

Fit an ordinary least-squares line $y = m x + b$ to those three points. The slope $m$ is the **local importance of $x$**. Recall the OLS slope formula:

$$ m = \frac{\sum_i (x_i - \bar{x})(y_i - \bar{y})}{\sum_i (x_i - \bar{x})^2} $$

Do it on paper, then run the cell.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# the three points we queried near x0 = 3
xs = np.array([2., 3., 4.])
ys = xs**2                      # black-box answers: 4, 9, 16

xbar, ybar = xs.mean(), ys.mean()
m = ((xs - xbar) * (ys - ybar)).sum() / ((xs - xbar)**2).sum()
b = ybar - m * xbar

print(f"local line:  y = {m:.2f} x + ({b:.2f})")
print(f"local importance of x (slope) = {m:.2f}")
print(f"calculus check: derivative of x^2 is 2x, and 2*3 = {2*3}")


## See it: the local line is tangent to the curve

The slope came out to **6** — exactly the derivative $2x$ at $x=3$. LIME *rediscovered the local slope of the model* just by querying it and fitting a line. Let's draw the black-box curve and our local explanation on top.

In [ ]:
grid = np.linspace(0, 6, 200)
plt.figure(figsize=(7, 4.5))
plt.plot(grid, grid**2, color='#1f77b4', lw=2, label='black box  f(x)=x²')
plt.scatter(xs, ys, color='black', zorder=5, label='queried points')
plt.plot(grid, m*grid + b, color='#d62728', ls='--', lw=2, label=f'LIME local line (slope={m:.0f})')
plt.scatter([3], [9], color='#d62728', s=120, zorder=6, edgecolor='k', label='point we explain (x₀=3)')
plt.ylim(-5, 36); plt.legend(); plt.title("LIME = fit a straight line to the black box, locally")
plt.xlabel("x"); plt.ylabel("prediction"); plt.tight_layout(); plt.show()


## Step 2 — *closeness matters*: the proximity weights

In the toy table our three points were all equally spaced, so a plain (unweighted) line was fine. But the real LIME insists that **points nearer to $x_0$ should count more** — after all, we only care about the model's behavior *right here*, not three blocks away.

So each perturbed point $z_i$ gets a **weight** from a *kernel* that decays with distance:

$$ w_i = \exp\!\left(-\frac{d(x_0, z_i)^2}{2\,\sigma^2}\right) $$

- $d(x_0, z_i)$ is the distance from the point we're explaining.
- $\sigma$ (the **kernel width**) is the single most important knob in LIME: small $\sigma$ = "only my immediate neighborhood"; large $\sigma$ = "a wide region" (more global, less local).

Then we fit a **weighted** least-squares line. Same idea as before, just with `sample_weight`. Let's generalize to real, multi-feature models.

## Step 3 — LIME from scratch on a REAL model (Boston Housing)

Same recipe as our permutation-importance, PDP, and SHAP notebooks: load Boston Housing, use the 4-feature "sneaky subset", min/max scale on train, fit a random forest. Then we'll write `lime_explain()` in about a dozen lines and explain one house.

In [ ]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.inspection import permutation_importance
from sklearn.metrics import r2_score

!gdown --id 1a0aNGSFWB-pf5ut1NsjE5ECIsbHHoAwI
df = pd.read_csv('BostonHousing.csv')

Y = df['medv']
feature_names = ['lstat', 'crim', 'dis', 'rad']      # same 4-feature subset as before
X = df[feature_names]

X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.2, shuffle=True, random_state=42)

scaler = MinMaxScaler()                                # scale on TRAIN only - no leakage
X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=feature_names, index=X_train.index)
X_test  = pd.DataFrame(scaler.transform(X_test),      columns=feature_names, index=X_test.index)

RFR = RandomForestRegressor(random_state=42).fit(X_train, y_train)
print("Test R2:", round(r2_score(y_test, RFR.predict(X_test)), 3))


In [ ]:
def lime_explain(model, x0, feature_names, perturb_sd=0.15, kernel_width=0.5,
                 n_samples=2000, seed=0):
    # LIME for one row x0 (features assumed scaled to roughly [0,1]).
    #   1) perturb x0 into a cloud of nearby fake rows
    #   2) get black-box predictions for the cloud
    #   3) weight each fake row by closeness to x0 (exponential kernel)
    #   4) fit a WEIGHTED linear regression -> coefficients are the explanation
    rng = np.random.RandomState(seed)
    x0  = np.asarray(x0, dtype=float)
    p   = len(feature_names)

    Z  = np.clip(x0 + rng.normal(0, perturb_sd, size=(n_samples, p)), 0, 1)  # 1) perturb
    yz = model.predict(Z)                                                    # 2) query

    d = np.sqrt(((Z - x0)**2).sum(axis=1))                                   # 3) weights
    w = np.exp(-(d**2) / (2 * kernel_width**2))

    surrogate = LinearRegression().fit(Z, yz, sample_weight=w)               # 4) local fit
    coefs = dict(zip(feature_names, surrogate.coef_))
    local_r2 = surrogate.score(Z, yz, sample_weight=w)        # how trustworthy is the local fit?
    return coefs, surrogate.intercept_, local_r2

# explain one test house
row_id = 0
x0 = X_test.iloc[row_id].values
coefs, intercept, local_r2 = lime_explain(RFR, x0, feature_names)

print(f"Explaining test house #{row_id}   (model predicts {RFR.predict(x0.reshape(1,-1))[0]:.2f})")
print(f"local surrogate R2 = {local_r2:.3f}  (closer to 1 = the local line fits well)\n")
for k in feature_names:
    arrow = "UP " if coefs[k] > 0 else "DOWN"
    print(f"  {k:>5}: local coefficient {coefs[k]:+.2f}   -> raising {k} pushes price {arrow}")


In [ ]:
# a horizontal bar chart of the local explanation (green = pushes price up, red = down)
fig, ax = plt.subplots(figsize=(7.5, 4))
order = sorted(feature_names, key=lambda k: coefs[k])
vals  = [coefs[k] for k in order]
colors = ['#2ca02c' if v > 0 else '#d62728' for v in vals]
ax.barh(order, vals, color=colors, edgecolor='black')
ax.axvline(0, color='black', lw=1)
ax.set_title(f"LIME explanation for test house #{row_id}")
ax.set_xlabel("local coefficient  (effect on predicted median value)")
plt.tight_layout(); plt.show()


## The knob that changes everything: kernel width

LIME's biggest gotcha: **the explanation depends on how far you "zoom."** A narrow kernel describes a tiny neighborhood; a wide kernel blends in far-away behavior and drifts toward a *global* linear model. Same house, different $\sigma$, different story. Always report your kernel width — and sanity-check the **local $R^2$** (a low value means a straight line just doesn't fit the black box here, so distrust the explanation).

In [ ]:
widths = [0.1, 0.25, 0.5, 1.0]
rows = []
for kw in widths:
    c, _, r2 = lime_explain(RFR, x0, feature_names, kernel_width=kw)
    rows.append({'kernel_width': kw, **{k: round(c[k], 2) for k in feature_names},
                 'local_R2': round(r2, 3)})
print(pd.DataFrame(rows).to_string(index=False))
print("\nNotice how the coefficients shift as the neighborhood widens - LIME is local!")


## ✏️ YOUR TURN — work a local slope by hand

Back to the whiteboard. Same black box $f(x) = x^2$, but now explain the prediction at $x_0 = 5$. Query three points and fit the OLS line **by hand**:

| perturbed $x$ | $y = f(x)$ |
|:---:|:---:|
| 4 | 16 |
| 5 | 25 |
| 6 | 36 |

1. Compute the slope $m$ with the OLS formula.
2. **Predict before you compute:** what should it be, from calculus? ($\frac{d}{dx}x^2 = 2x$.)
3. Check yourself in the cell, then explain in one sentence why a *narrower* perturbation (say 4.9, 5.0, 5.1) would give an even more accurate local slope.

In [ ]:
# ✏️ YOUR TURN: fill in the queried points, compute the local slope.
xs2 = np.array([4., 5., 6.])      # <-- the perturbed x values
ys2 = xs2**2                      # <-- black-box answers

xbar2, ybar2 = xs2.mean(), ys2.mean()
m2 = ((xs2 - xbar2) * (ys2 - ybar2)).sum() / ((xs2 - xbar2)**2).sum()
print("local slope at x0=5:", round(m2, 3), " | calculus says 2*5 =", 2*5)


```{admonition} 🔑 Solution (instructor — delete before sharing with students)
:class: dropdown
$\bar{x}=5,\ \bar{y}=\tfrac{77}{3}\approx 25.67$. Then $m = \dfrac{(-1)(16-25.67) + 0 + (1)(36-25.67)}{1+0+1} = \dfrac{20}{2} = \boxed{10}$, exactly $2\times 5$.

A **narrower** perturbation hugs the curve more tightly, so the straight-line approximation has less curvature to ignore — the local slope gets closer to the true derivative. That is the kernel-width trade-off in miniature.
```

## LIME vs SHAP — same question, different philosophies

Both explain a *single* prediction, locally. But they are **not** the same number, because they answer subtly different questions:

| | **SHAP** (last lecture) | **LIME** (this lecture) |
|---|---|---|
| Core idea | fair credit via game theory | fit a local linear surrogate |
| Guarantees | adds up to the prediction exactly; consistent | no additivity guarantee |
| Main knob | choice of **background** | choice of **kernel width** |
| Cost | exponential in features (needs shortcuts) | a handful of `predict` calls — **fast** |
| Watch out for | background choice | instability / kernel sensitivity |

Let's put them side by side for the same house. We'll reuse the exact-Shapley function from the SHAP notebook. They won't match to the decimal — but they should **agree on direction and rough magnitude**.

In [ ]:
import itertools, math
def shapley_from_scratch(model, x_row, background, feature_names):
    x_row = np.asarray(x_row, float); bg = np.asarray(background, float)
    p = len(feature_names); idx = list(range(p))
    def v(S):
        S = set(S); Z = bg.copy()
        for j in S: Z[:, j] = x_row[j]
        return model.predict(Z).mean()
    phi = {}
    for i in idx:
        others = [j for j in idx if j != i]; tot = 0.0
        for k in range(len(others)+1):
            for S in itertools.combinations(others, k):
                w = math.factorial(len(S))*math.factorial(p-len(S)-1)/math.factorial(p)
                tot += w*(v(set(S)|{i}) - v(set(S)))
        phi[feature_names[i]] = tot
    return phi

background = X_train.sample(100, random_state=0).values
shap_vals = shapley_from_scratch(RFR, x0, background, feature_names)
lime_vals, _, _ = lime_explain(RFR, x0, feature_names)

xpos = np.arange(len(feature_names)); width = 0.38
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(xpos - width/2, [shap_vals[k] for k in feature_names], width, label='SHAP value', color='#1f77b4')
ax.bar(xpos + width/2, [lime_vals[k] for k in feature_names], width, label='LIME coefficient', color='#ff7f0e')
ax.axhline(0, color='black', lw=1); ax.set_xticks(xpos); ax.set_xticklabels(feature_names)
ax.set_title(f"SHAP vs LIME for test house #{row_id}  (same direction, different math)")
ax.legend(); plt.tight_layout(); plt.show()


## Sanity check: the `lime` library agrees

Everything above used only `numpy` and one `LinearRegression`. The `lime` package does exactly this — perturb, weight, fit a local linear model — with extra polish. Let's confirm it tells the same story.

In [ ]:
try:
    import lime, lime.lime_tabular
except ImportError:
    !pip -q install lime
    import lime, lime.lime_tabular

explainer = lime.lime_tabular.LimeTabularExplainer(
    X_train.values, feature_names=feature_names, mode='regression',
    discretize_continuous=False, random_state=0)

exp = explainer.explain_instance(x0, RFR.predict, num_features=4)
print("lime library's local explanation:")
for feat, weight in exp.as_list():
    print(f"   {feat:>8}: {weight:+.2f}")
print("\nOur from-scratch coefficients:")
for k in feature_names:
    print(f"   {k:>8}: {lime_vals[k]:+.2f}")


## Wrap-up

```{admonition} Key takeaways
:class: tip
- **LIME explains ONE prediction** by fitting a simple, transparent model (a weighted linear regression) to the black box **in a small neighborhood** of that point.
- The recipe: **perturb → predict → weight by closeness → fit a local line → read the coefficients.**
- On a 1-D curve, LIME literally recovers the **local slope (derivative)** — we computed it by hand and it matched $2x$.
- The **kernel width** is the master knob: too wide and the explanation goes global; always report it and check the **local $R^2$** for fidelity.
- **SHAP vs LIME**: SHAP gives fair, additive, consistent credit but is expensive; LIME is fast and intuitive but kernel-sensitive and not additive. Use both — when they *agree*, trust the story.
- Model-agnostic to the core: we only ever called `model.predict()`.
```

That closes our explainability toolkit: **permutation importance** (which features), **PDP/ICE** (how, globally), **SHAP** (fair local credit), and **LIME** (local linear surrogate). You can now open *any* black box and tell a defensible story about it.

---
*Dr. Dave Wanik — OPIM 5512, University of Connecticut.*